# Diarization validation (dev-only)

Compares `services/audio/run.py`'s diarized output
(`db/blob_store/{patient_id}/diarized_transcript.json`) against the TTS-generation-time
ground truth (`notebooks/ground_truth/{patient_id}_transcript.json`), using
`validate_diarization` from `services/audio/src/diarization_utils.py`.

Production code never reads this ground truth - it's a one-off dev-time check.

Run `make generate-dataset` then `python services/audio/run.py` first so
`db/blob_store/` is populated, then run this notebook.

In [11]:
import json
import sys
from pathlib import Path

NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
sys.path.insert(0, str(REPO_ROOT / "services" / "audio" / "src"))

from diarization_utils import validate_diarization

BLOB_STORE_DIR = REPO_ROOT / "db" / "blob_store"
GROUND_TRUTH_DIR = REPO_ROOT / "notebooks" / "ground_truth"

In [12]:
participant_ids = sorted(p.stem.replace("_transcript", "") for p in GROUND_TRUTH_DIR.glob("*_transcript.json"))
print(f"{len(participant_ids)} participants with ground truth: {participant_ids}")

14 participants with ground truth: ['OAW01', 'OAW02', 'OAW03', 'OAW04', 'OAW05', 'OAW06', 'OAW07', 'OAW08', 'OAW09', 'OAW10', 'OAW11', 'OAW12', 'OAW13', 'OAW14']


In [13]:
results = []
for participant_id in participant_ids:
    diarized_path = BLOB_STORE_DIR / participant_id / "diarized_transcript.json"
    if not diarized_path.exists():
        print(f"{participant_id}: no diarized_transcript.json - run services/audio/run.py first")
        continue

    ground_truth_log = json.loads(
        (GROUND_TRUTH_DIR / f"{participant_id}_transcript.json").read_text(encoding="utf-8")
    )
    segments = json.loads(diarized_path.read_text(encoding="utf-8"))["segments"]

    validation = validate_diarization(segments, ground_truth_log)
    validation["participant_id"] = participant_id
    results.append(validation)

In [14]:
if not results:
    print(
        "No results to summarize - either notebooks/ground_truth/*_transcript.json is "
        "empty (run `make generate-dataset` to populate it) or none of those "
        "participants have a db/blob_store/{participant_id}/diarized_transcript.json "
        "yet (run `python services/audio/run.py`)."
    )
else:
    print(f"{'participant':<12}{'accuracy':<10}{'diarized':<10}{'ground truth':<14}")
    for r in results:
        print(f"{r['participant_id']:<12}{r['turn_sequence_accuracy']:<10.2f}{r['num_diarized_turns']:<10}{r['num_ground_truth_turns']:<14}")

    perfect = sum(1 for r in results if r["turn_sequence_accuracy"] == 1.0)
    mean_accuracy = sum(r["turn_sequence_accuracy"] for r in results) / len(results)
    print(f"\n{perfect}/{len(results)} participants fully correct, mean turn-sequence accuracy {mean_accuracy:.3f}")

participant accuracy  diarized  ground truth  
OAW01       1.00      7         7             
OAW02       1.00      7         7             
OAW03       1.00      7         7             
OAW04       1.00      9         9             
OAW05       0.86      6         7             
OAW06       1.00      7         7             
OAW07       0.86      6         7             
OAW08       1.00      7         7             
OAW09       1.00      6         6             
OAW10       1.00      6         6             
OAW11       1.00      5         5             
OAW12       1.00      9         9             
OAW13       1.00      6         6             
OAW14       1.00      7         7             

12/14 participants fully correct, mean turn-sequence accuracy 0.980


In [15]:
for r in results:
    if r["turn_sequence_accuracy"] < 1.0:
        print(f"{r['participant_id']}:")
        print(f"  diarized:      {r['diarized_sequence']}")
        print(f"  ground truth:  {r['ground_truth_sequence']}")

OAW05:
  diarized:      ['doctor', 'patient', 'doctor', 'patient', 'doctor', 'patient']
  ground truth:  ['doctor', 'patient', 'doctor', 'patient', 'doctor', 'patient', 'doctor']
OAW07:
  diarized:      ['doctor', 'patient', 'doctor', 'patient', 'doctor', 'patient']
  ground truth:  ['doctor', 'patient', 'doctor', 'patient', 'doctor', 'patient', 'doctor']
